> This is a self-correcting activity generated by [nbgrader](https://nbgrader.readthedocs.io). Fill in any place that says `YOUR CODE HERE` or `YOUR ANSWER HERE`. Run subsequent cells to check your code.

---

# Breast cancer

In this activity, you'll use a K-Nearest Neighbors classifier to help diagnose breast tumors.

The [Breast Cancer][1] dataset is used for multivariate binary classification between benign and maligant tumors. There are 569 total samples with 30 features each. Features were computed from a digitized image of a fine needle aspirate of a breast mass. They describe characteristics of the cell nuclei present in the image.

![Breast cancer logo](https://github.com/bpesquet/mlkatas/blob/master/training/images/breast-cancer.jpg?raw=1)

[1]: https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)

## Environment setup

In [14]:
# Import base packages
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [15]:
# Setup plots
%matplotlib inline
plt.rcParams['figure.figsize'] = 10, 8
%config InlineBackend.figure_format = 'retina'
sns.set()

In [44]:
# Import ML packages
import sklearn
print(f'scikit-learn version: {sklearn.__version__}')

from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.metrics import classification_report # Importing classification_report from sklearn.metrics


scikit-learn version: 1.6.0


## Step 1: Loading the data

In [13]:
dataset = load_breast_cancer()

# Put data in a pandas DataFrame
df_breast_cancer = pd.DataFrame(dataset.data, columns=dataset.feature_names)
# Add target and class to DataFrame
df_breast_cancer['target'] = dataset.target
df_breast_cancer['class'] = dataset.target_names[dataset.target]
# Show 10 random samples
df_breast_cancer.sample(n=10)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,class
392,15.49,19.97,102.40,744.7,0.11600,0.15620,0.18910,0.09113,0.1929,0.06744,...,142.10,1359.0,0.1681,0.39130,0.55530,0.21210,0.3187,0.10190,0,malignant
237,20.48,21.46,132.50,1306.0,0.08355,0.08348,0.09042,0.06022,0.1467,0.05177,...,161.70,1750.0,0.1228,0.23110,0.31580,0.14450,0.2238,0.07127,0,malignant
218,19.80,21.56,129.70,1230.0,0.09383,0.13060,0.12720,0.08691,0.2094,0.05581,...,170.30,2009.0,0.1353,0.32350,0.36170,0.18200,0.3070,0.08255,0,malignant
164,23.27,22.04,152.10,1686.0,0.08439,0.11450,0.13240,0.09702,0.1801,0.05553,...,184.20,2403.0,0.1228,0.35830,0.39480,0.23460,0.3589,0.09187,0,malignant
472,14.92,14.93,96.45,686.9,0.08098,0.08549,0.05539,0.03221,0.1687,0.05669,...,112.00,906.6,0.1065,0.27910,0.31510,0.11470,0.2688,0.08273,1,benign
174,10.66,15.15,67.49,349.6,0.08792,0.04302,0.00000,0.00000,0.1928,0.05975,...,73.20,408.3,0.1076,0.06791,0.00000,0.00000,0.2710,0.06164,1,benign
479,16.25,19.51,109.80,815.8,0.10260,0.18930,0.22360,0.09194,0.2151,0.06578,...,122.10,939.7,0.1377,0.44620,0.58970,0.17750,0.3318,0.09136,0,malignant
143,12.90,15.92,83.74,512.2,0.08677,0.09509,0.04894,0.03088,0.1778,0.06235,...,97.17,643.8,0.1312,0.25480,0.20900,0.10120,0.3549,0.08118,1,benign
76,13.53,10.94,87.91,559.2,0.12910,0.10470,0.06877,0.06556,0.2403,0.06641,...,91.36,605.5,0.1451,0.13790,0.08539,0.07407,0.2710,0.07191,1,benign
96,12.18,17.84,77.79,451.1,0.10450,0.07057,0.02490,0.02941,0.1900,0.06635,...,82.14,495.2,0.1140,0.09358,0.04980,0.05882,0.2227,0.07376,1,benign


## Step 2: Preparing the data

### Question

Compute the number of features of the dataset into the `num_features` variable.

In [20]:
num_features = df_breast_cancer.shape[1]


In [21]:
print(f'Number of features: {num_features}')

assert num_features == 32

Number of features: 32


### Question

In order to evaluate class distribution, compute the number of benign and malignant tumors into the `num_benign` and `num_malignant` variables respectively.

In [23]:
num_benign = df_breast_cancer["class"].value_counts()[0]  # Conta i benigni (0)
num_malignant = df_breast_cancer["class"].value_counts()[1]  # Conta i maligni (1)


<ipython-input-23-154f6224013f>:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  num_benign = df_breast_cancer["class"].value_counts()[0]  # Conta i benigni (0)
<ipython-input-23-154f6224013f>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  num_malignant = df_breast_cancer["class"].value_counts()[1]  # Conta i maligni (1)


In [24]:
print(f'Benign count: {num_benign}. Malignant count: {num_malignant}')

assert num_benign == 357
assert num_malignant == 212

Benign count: 357. Malignant count: 212


In [25]:
# Store input and labels
x = dataset.data
y = dataset.target

print(f'x: {x.shape}. y: {y.shape}')

x: (569, 30). y: (569,)


### Question

Split the dataset into training and test sets with a 25% ratio. Use variables `x_train`, `y_train`, `x_test` and `y_test`.

In [32]:
# Separa le caratteristiche (X) e la variabile target (y)
X = df_breast_cancer.drop(columns=['class','target'])
y = df_breast_cancer['class']  # La colonna target
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [33]:
print(f'x_train: {x_train.shape}. y_train: {y_train.shape}')
print(f'x_test: {x_test.shape}. y_test: {y_test.shape}')

assert x_train.shape == (426, 30)
assert y_train.shape == (426, )
assert x_test.shape == (143, 30)
assert y_test.shape == (143,)

x_train: (426, 30). y_train: (426,)
x_test: (143, 30). y_test: (143,)


### Question

Scale features by standardization while preventing information leakage from the test set.

In [36]:
# Inizializza lo scaler
scaler = StandardScaler()

# Calcola la media e la deviazione standard sul training set e trasforma il training set
x_train = scaler.fit_transform(x_train)

# Applica la trasformazione (senza ricalcolare la media e la deviazione standard) sul test set
x_test = scaler.transform(x_test)

In [37]:
mean_train = x_train.mean()
std_train = x_train.std()

print(f'mean_train: {mean_train}. std_train: {std_train}')

assert np.abs(np.max(mean_train)) < 10**-6
assert np.abs(np.max(std_train - 1)) < 10**-6

mean_train: -5.337410221672114e-16. std_train: 0.9999999999999999


## Step 3: Creating a classifier

### Question

Create a `KNeighborsClassifier` instance using only one nearest neighbor, store it into the `model` variable, and fit the training data.

In [38]:
# Crea un'istanza del modello KNN con un solo vicino
model = KNeighborsClassifier(n_neighbors=1)

# Addestra il modello sul set di training
model.fit(x_train, y_train)

# Conferma che il modello è stato addestrato
print("Modello KNN addestrato con successo.")

Modello KNN addestrato con successo.


## Step 4: Evaluating the classifier

In [39]:
# Compute accuracy on training and test sets
train_acc = model.score(x_train, y_train)
test_acc = model.score(x_test, y_test)

print(f'Training accuracy: {train_acc * 100:.2f}%')
print(f'Test accuracy: {test_acc * 100:.2f}%')

Training accuracy: 100.00%
Test accuracy: 95.10%


### Question

Display precision, recall and f1-score for the classifier on test data. Interpret the results.

In [45]:
precision = metrics.precision_score(y_test, model.predict(x_test), pos_label='malignant')
recall = metrics.recall_score(y_test, model.predict(x_test), pos_label='malignant')
f1 = metrics.f1_score(y_test, model.predict(x_test), pos_label='malignant')

print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1-score: {f1 * 100:.2f}%')

print(classification_report(y_test, model.predict(x_test)))


Precision: 94.34%
Recall: 92.59%
F1-score: 93.46%
              precision    recall  f1-score   support

      benign       0.96      0.97      0.96        89
   malignant       0.94      0.93      0.93        54

    accuracy                           0.95       143
   macro avg       0.95      0.95      0.95       143
weighted avg       0.95      0.95      0.95       143



### Question

Go back to step 3 and try to find the best value for the `k` number of nearest neighbors.